# RAG 1｜血染鐘樓《規則怪談》知識庫：EmbeddingGemma 2 + FAISS

> ✏️ **本作業主題**：把血染鐘樓（Blood on the Clocktower）自製劇本《規則怪談》的介紹文 PDF，整理成 **69 個 chunk**（小鎮規則手冊 16 條、27 個角色能力、9 條相剋／特殊規則、劇本評析、說書人手冊等），建立向量資料庫，給第二份 Colab 的「鴉木布拉夫小鎮祭司」對話機器人使用。

以老師的範例程式為底修改；**每個 chunk 產生一個向量**，建立 FAISS 精確搜尋索引並匯出 `rules_horror_vector_db.zip`。

**操作方式：請由上往下，一格一格執行（或直接「全部執行」）。**

流程：GitHub 下載資料 ZIP → 擷取 RAG 文字與 metadata → embedding → FAISS → 用問題集驗證檢索 → 匯出知識庫。

本程式不呼叫 LLM，不需要 Groq 密鑰。

### 資料來源與 chunk 設計
- 來源：`【染鐘樓】劇本社群—第332期《規則怪談》.pdf`（鐘樓劇本博物館，劇本作者：星火樂）。
- PDF 文字層（規則手冊、評析）用 pypdf 擷取；角色表與人數配置表是**圖片**，由 AI 協助辨識並轉成繁體，再人工核對。
- 每個 TXT = 一條規則／一個角色／一條相剋規則／一個評析段落，欄位如下：

| 欄位 | 用途 |
|---|---|
| 標題、【內容】 | **送進 embedding**，也會原樣提供給 LLM |
| 資料類型、分類、陣營與角色類型、來源文件、頁碼、原始出處、整理日期 | metadata：顯示來源、追溯頁碼 |
| 【檢索練習問題】 | 不進 embedding；彙整在 `問題集.csv`，用來驗證檢索 |
| 【提醒】 | 不進 embedding |

## 1. 安裝套件

我們使用 Sentence Transformers 載入模型、NumPy 處理向量，以及 FAISS 搜尋向量。這格固定本教材採用的套件版本，方便記錄與重現。

先執行本格，再執行下格的 import。如果你已在本次執行階段載入其他版本的相關套件，請在安裝後重新啟動工作階段，再從下一格繼續；不需要重跑安裝格。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 檢查環境與集中設定

`BATCH_SIZE` 可以調小以降低記憶體使用量。`EXPECTED_DOCUMENT_COUNT` 只用來提醒教材篇數；換自己的資料時可設為 `None`，不會強制只能有16篇。

以下會列出套件版本和 CPU／GPU。Colab 常見的 T4 不支援原生 bfloat16，本程式會改用 float32；不使用 EmbeddingGemma 2 不相容的 float16。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = 69  # ✏️ 《規則怪談》資料集共 69 個 chunk
EXPORT_ZIP_NAME = "rules_horror_vector_db.zip"  # ✏️ 改成本主題的向量資料庫名稱

# ✏️ 原始 TXT 資料集放在 GitHub，助教不用手動上傳；網址失效時才改用上傳。
DATA_ZIP_URL = "https://raw.githubusercontent.com/Wilson0503/rules_horror_rag-data/main/rules_horror_rag-data.zip"

# text 本身已含標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

## 3. ✏️ 從 GitHub 下載《規則怪談》資料 ZIP

範例原本要手動上傳，這裡改成**先從 `DATA_ZIP_URL` 自動下載** `rules_horror_rag-data.zip`；下載失敗才跳出上傳視窗。

程式只解開 TXT；README、`sources.csv`、`問題集.csv` 不會匯入知識庫（問題集會另外讀出來做檢索驗證）。

In [ ]:
import io
import urllib.request

def download_bytes(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read()

try:
    archive_bytes = download_bytes(DATA_ZIP_URL)
    archive_name = DATA_ZIP_URL.rsplit("/", 1)[-1]
    print("已從 GitHub 下載：", DATA_ZIP_URL)
except Exception as exc:
    print("GitHub 下載失敗（", type(exc).__name__, "），改用手動上傳。")
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("請一次上傳一個資料 ZIP 檔。")
    archive_name = zip_names[0]
    archive_bytes = uploaded[archive_name]

SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
DATA_DIR = Path(tempfile.mkdtemp(prefix="rag_documents_"))

# 僅解開 TXT，並確保 ZIP 中的路徑不會寫到暫存資料夾外。
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [
        member for member in archive.infolist()
        if not member.is_dir()
        and member.filename.lower().endswith(".txt")
        and not any(part.startswith((".", "__MACOSX")) for part in Path(member.filename).parts)
    ]
    if not members:
        raise ValueError("ZIP 中找不到 TXT；請確認資料集格式。")
    destinations = set()
    for member in members:
        target = (DATA_DIR / member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            raise ValueError(f"ZIP 包含不適當的檔案路徑：{member.filename}")
        if target in destinations:
            raise ValueError(f"ZIP 有重複檔名：{member.filename}")
        destinations.add(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
    # ✏️ 另外讀出問題集，稍後驗證檢索效果（不放進知識庫）。
    question_names = [n for n in archive.namelist() if n.endswith("問題集.csv")]
    question_set = (
        pd.read_csv(io.BytesIO(archive.read(question_names[0])), encoding="utf-8-sig")
        if question_names else None
    )

txt_paths = sorted(
    [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".txt"],
    key=lambda p: p.relative_to(DATA_DIR).as_posix(),
)
print("資料集：", archive_name)
print("TXT 數量：", len(txt_paths))
for path in txt_paths:
    print("•", path.relative_to(DATA_DIR).as_posix())
if EXPECTED_DOCUMENT_COUNT is not None and len(txt_paths) != EXPECTED_DOCUMENT_COUNT:
    print(f"提醒：本次為 {len(txt_paths)} 篇，與預期的 {EXPECTED_DOCUMENT_COUNT} 篇不同；仍可繼續。")
print("問題集題數：", 0 if question_set is None else len(question_set))

## 4. 先看一個原始 chunk

請觀察「標題」「分類」「陣營與角色類型」「頁碼」「【內容】」「【檢索練習問題】」等欄位。不是每個欄位都適合送進 embedding。

只取標題與正文。若把練習問題也送進去，搜尋會直接命中題目，無法公平觀察正文的檢索效果。

可修改 `SAMPLE_INDEX`，例如 `20` 是第一個角色「貴族」、`47` 是第一條相剋規則。

In [ ]:
SAMPLE_INDEX = 0
sample_path = txt_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

## 5. 第一個可修改函式：取出 RAG 文字

`extract_rag_text(raw_text)` 回傳「標題＋【內容】正文」。找到【內容】後，讀到下一個【區段標題】為止；練習問題和提醒不會混入。

✏️ 《規則怪談》的 TXT 沿用範例的「標題：」與「【內容】」格式，所以擷取規則可以直接沿用。角色 chunk 的正文開頭已寫上「角色名（陣營·類型）」，讓「提線木偶是爪牙嗎？」這類問題也能檢索到。

這個回傳值會原樣保存；未來檢索後也將它交給 LLM，確保搜尋與回答使用同一份內容。

In [ ]:
def extract_rag_text(raw_text):
    """從本教材 TXT 取出標題與正文；換資料時修改這裡。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    title_match = re.search(r"^標題[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    content_match = re.search(
        r"^【內容】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
        normalized,
        flags=re.MULTILINE | re.DOTALL,
    )
    if content_match is None or not content_match.group("body").strip():
        raise ValueError("找不到非空的【內容】正文；請修改 extract_rag_text() 以配合文件格式。")
    title = title_match.group(1).strip() if title_match else ""
    body = content_match.group("body").strip()
    return f"{title}\n\n{body}" if title else body

sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))

## 6. ✏️ 第二個可修改函式：取出 metadata

`extract_metadata(raw_text, filename)` 回傳追溯資料用的資訊，不會送進 embedding。

依《規則怪談》資料集改成以下欄位：`title`、`category`（劇本資訊／小鎮規則手冊／角色能力／相剋規則／劇本評析／說書人手冊）、`team`（陣營與角色類型）、`page`（PDF 頁碼）、`source_file`、`origin`、`date`、`filename`、`document_type`。

本資料來自 PDF，沒有網址，所以 `source_url` 設為 `None`，不自行捏造；第二份 Colab 改用「PDF 頁碼」作為來源顯示。

In [ ]:
def extract_metadata(raw_text, filename):
    """取出《規則怪談》chunk 的描述資訊。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    def read_field(name):
        match = re.search(rf"^{re.escape(name)}[：:][ \t]*(.*)$", normalized, flags=re.MULTILINE)
        return match.group(1).strip() or None if match else None

    return {
        "title": read_field("標題") or Path(filename).stem,
        "category": read_field("分類") or "未分類",
        "team": read_field("陣營與角色類型"),
        "page": read_field("頁碼"),
        "source_file": read_field("來源文件"),
        "source_url": None,  # PDF 資料沒有網址，不捏造。
        "origin": read_field("原始出處"),
        "date": read_field("整理日期"),
        "filename": filename,
        "document_type": read_field("資料類型") or "未標示",
    }

sample_metadata = extract_metadata(sample_raw, sample_path.relative_to(DATA_DIR).as_posix())
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))

## 7. 批次整理成統一格式

把兩個函式組合起來，每篇得到 `doc_id`、`text` 和 `metadata`。檔案先排序，再按順序給編號；只要教材及排序不變，編號便相同。新增文件可能改變編號，因此需要整套重建索引與對照。

請檢查列表中每篇是否有標題、正文及來源。缺少 metadata 會提醒；正文缺失則停止並指出檔名，讓你回到前兩格修改函式。

In [ ]:
documents = []
for number, path in enumerate(txt_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")
        rag_text = extract_rag_text(raw_text)
        metadata = extract_metadata(raw_text, filename)
    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc
    documents.append({"doc_id": f"doc_{number:03d}", "text": rag_text, "metadata": metadata})

if not documents:
    raise ValueError("沒有可建立索引的文件。")
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "標題": doc["metadata"]["title"], "分類": doc["metadata"]["category"],
     "正文含標題字元數": len(doc["text"]), "頁碼": doc["metadata"]["page"]}
    for doc in documents
])
display(summary)
display(summary["分類"].value_counts().rename("chunk 數"))
for doc in documents:
    missing = [key for key in ("page", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))
print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

## 8. 載入 EmbeddingGemma 2

先查詢模型目前的確切版本（commit），再載入該版本，將版本記錄到輸出設定，方便第二份 Colab 使用相同模型。只啟用文字部分，使用適合裝置的精度。

首次下載需要等待。本步使用公開模型，不讀取 Groq 或其他密鑰。若下載失敗，先確認網路；模型架構不受支援時，確認安裝格已完成，並重新啟動工作階段。程式不會默默改用別的模型。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

## 9. 檢查輸入長度，避免靜默截斷

本模型採用不同的任務前綴來區分文件和問題。文件本身已有標題，所以組成 `title: none | text: {text}`；問題則用 `task: search result | query: {question}`。

下格計算實際送入模型的文件文字（含前綴）長度。超過上限便停止，不會只取前半段。換成長文件時，可以在後續課程加入 chunking，或重新設計文件單位。

In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("本教材使用的 token 上限：", TOKEN_LIMIT)
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])





## 10. 產生並觀察文件向量

一篇文件對應矩陣中的一列，欄數是向量維度。`prompt=""` 表示前綴已自行加入，不讓模型額外再加一次。

向量正規化後長度約為1，內積便等於 cosine similarity。下格也將資料轉為 FAISS 使用的 float32 連續陣列，檢查空向量、非有限值及形狀。

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

## 11. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 比較問題向量與所有文件向量的內積。因為雙方都正規化，排名就是 cosine similarity 的排名。

FAISS 回傳的是向量編號，**不是原文**。`index_doc_ids` 保存向量順序與文件編號的對應，再透過 `doc_id` 找回正文與 metadata。

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

## 12. 定義問題向量化與搜尋函式

問題也要用同一個 embedding 模型向量化，但使用查詢專用前綴。`top_k` 表示取回幾篇文件；程式限制它不超過實際文件數。

搜尋函式只做檢索，還沒有呼叫 LLM。精確搜尋保證依目前向量找到分數最高者，不保證文件一定足以回答問題。

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 13. ✏️ 試查一次，閱讀真正取回的內容

修改 `QUESTION` 和 `TOP_K`，重新執行本格即可查詢。先看標題及分數，再讀完整正文，判斷是否有足夠依據。

可以比較 `top_k=1`、`3`、`5`。也可問「鎮上的祭司叫什麼名字？」：即使資料沒有答案，FAISS 仍會傳回最相近的 chunk。**相似度不是正確率，也不是模型有答案的機率。**

In [ ]:
QUESTION = "提線木偶和告密者同時在場時，誰會得知三個不在場的角色？"
TOP_K = 3
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([
    {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
     "cosine_similarity": r["score"], "分類": r["metadata"]["category"], "頁碼": r["metadata"]["page"]}
    for r in results
]))
for result in results:
    print(f"\n[{result['doc_id']}] {result['metadata']['title']} ({result['score']:.4f})")
    print(result["text"])
    print("來源：PDF 第", result["metadata"]["page"] or "?", "頁")

## 13-1. ✏️ 新增：用問題集批次驗證檢索效果

每個 chunk 都有一題【檢索練習問題】（彙整在 `問題集.csv`），這題沒有送進 embedding。這裡把每題拿去搜尋，看「預期的 chunk」有沒有排在第 1 名（Hit@1）或前 3 名（Hit@3）。

跨文件問題列出多個預期檔案，只要命中其中一個就算；「應拒絕臆測」的題目沒有正確檔案，不計分，留給第二份 Colab 觀察 LLM 會不會拒答。

In [ ]:
if question_set is None:
    print("資料 ZIP 中沒有問題集.csv，略過本格。")
else:
    filename_to_doc = {doc["metadata"]["filename"]: doc["doc_id"] for doc in documents}
    stem_to_doc = {Path(name).stem.split("_", 1)[1]: doc_id for name, doc_id in filename_to_doc.items()}
    rows = []
    for _, row in question_set.iterrows():
        if row["預期相關檔案"] == "無":
            continue
        expected = set()
        for name in str(row["預期相關檔案"]).split("/"):
            name = name.strip()
            expected.add(filename_to_doc.get(name) or stem_to_doc.get(name))
        hits = [r["doc_id"] for r in search_documents(row["測試問題"], 3)]
        rows.append({
            "問題": row["測試問題"], "類型": row["類型"],
            "第1名": documents_by_id[hits[0]]["metadata"]["title"],
            "Hit@1": hits[0] in expected, "Hit@3": bool(expected & set(hits)),
        })
    evaluation = pd.DataFrame(rows)
    print(f"共 {len(evaluation)} 題；Hit@1 = {evaluation['Hit@1'].mean():.1%}，Hit@3 = {evaluation['Hit@3'].mean():.1%}")
    display(evaluation[~evaluation["Hit@3"]].rename_axis("未命中前3名的題目"))

## 14. 保存索引、文件、編號對照與設定

第二份 Colab 需要下面四個檔案，不能只拿 FAISS 索引：

| 檔案 | 用途 |
|---|---|
| `index.faiss` | 文件向量與搜尋索引 |
| `documents.json` | 取回後提供給 LLM 的正文與來源資訊 |
| `index_doc_ids.json` | FAISS 編號對應的文件編號 |
| `config.json` | 模型確切版本、前綴、維度、正規化、套件版本及檢查資訊 |

不保存模型權重、API 金鑰或其他環境變數。第二份 Colab 載入相同模型版本，只重新計算問題向量。

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "knowledge_base": "血染鐘樓《規則怪談》",  # ✏️ 本主題
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_chunk_one_vector",  # ✏️ 一條規則／一個角色＝一個向量
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")

## 15. 重新載入，驗證第二份 Colab 所需的資料

從剛才保存的檔案重新載入，檢查完整性、文件編號與向量對應，再使用**同一個問題向量**比對保存前後的排名與分數。此處不重算問題向量，以免不同運算裝置的細微浮點誤差干擾索引保存檢查。

也將 FAISS 結果與 NumPy 的全矩陣內積排序比較，確認精確搜尋的意義。遇到完全同分時，兩套工具的同分排列順序可以不同。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

## 16. ✏️ 打包並下載知識庫，上傳到 GitHub

ZIP 根目錄包含前述四個檔案，檔名為 **`rules_horror_vector_db.zip`**。

下載後請**上傳到 GitHub repo**（與 `rules_horror_rag-data.zip` 放在同一個資料夾），第二份 Colab 會用 raw 網址自動下載，助教不必手動上傳。

In [ ]:
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")

In [ ]:
from google.colab import files
files.download(str(EXPORT_ZIP))

## 17. 回顧：本作業相對於範例的修改

1. **資料**：改為血染鐘樓《規則怪談》劇本 PDF，切成 69 個 TXT chunk（`build_chunks.py` 產生）。
2. **讀取資料**：改成從 GitHub 自動下載資料 ZIP（失敗才手動上傳）。
3. **`extract_metadata()`**：改成分類、陣營、PDF 頁碼等欄位；PDF 沒有網址，不捏造 `source_url`。
4. **檢索驗證**：新增 13-1，用問題集計算 Hit@1／Hit@3。
5. **輸出檔名**：`rules_horror_vector_db.zip`。

**觀察問題：** 「提線木偶」相關的問題常同時命中角色 chunk 與多條相剋規則 chunk，`top_k=3` 比 `top_k=1` 更能把相剋規則一起帶給 LLM。

### 參考資料

- [EmbeddingGemma 2 官方 Sentence Transformers 指引](https://ai.google.dev/gemma/docs/embeddinggemma/inference-embeddinggemma-with-sentence-transformers)
- [EmbeddingGemma 2 模型](https://huggingface.co/google/embeddinggemma-2)
- [FAISS 入門與索引說明](https://github.com/facebookresearch/faiss/wiki/Getting-started)